In [5]:
import sys
if "google.colab" in sys.modules:
    print("estas en colab")

estas en colab


In [7]:
!git status

fatal: not a git repository (or any of the parent directories): .git


In [6]:
from google.colab import drive
drive.mount('/content/drive')

# Agrega la carpeta al path
import sys
sys.path.append('/content/drive/My Drive/tu-carpeta-en-drive')

import pandas as pd
from constants import MAX_SEQ_LEN, DATOS_PATH
from collections import Counter

KeyboardInterrupt: 

In [3]:
%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'imp'

In [2]:
import pandas as pd
from constants import MAX_SEQ_LEN, DATOS_PATH
from collections import Counter

ModuleNotFoundError: No module named 'constants'

In [3]:
df = pd.read_csv(DATOS_PATH, sep="\t", header=None, encoding="utf-8")
df = df.drop(columns=[0, 2])
df = df.rename(columns={1: "spanish", 3: "english"})
df.tail()

In [4]:
df.info()

### Traductor

In [5]:
spanish = df["spanish"].tolist()
english = df["english"].tolist()

In [6]:
print(spanish[:3])
print(english[:3])

In [7]:
def preprocesar_enunciado(enunciado: str):
    enunciado = str(enunciado)
    enunciado = enunciado.lower().strip()
    # enunciado = enunciado.replace(".", " .")
    # enunciado = enunciado.replace(",", " ,")
    # enunciado = enunciado.replace("¿", " ¿ ")
    # enunciado = enunciado.replace("?", " ? ")
    # enunciado = enunciado.replace("¡", " ¡ ")
    # enunciado = enunciado.replace("!", " ! ")
    enunciado = '<sos> ' + enunciado + ' <eos>'
    return enunciado

In [8]:
enunciado = "¡Hola! ¿Cómo estás?"
print(preprocesar_enunciado(enunciado))

In [9]:
spanish_preprocesado = [preprocesar_enunciado(s) for s in spanish]
english_preprocesado = [preprocesar_enunciado(e) for e in english]

In [10]:
print(spanish_preprocesado[0:3])
print(english_preprocesado[0:3])

In [11]:
def construir_palabras(enunciados):
    palabras = [palabra for enunciado in enunciados for palabra in enunciado.split(" ")]
    palabras_count = Counter(palabras)
    palabras_ordenadas = sorted(palabras_count.items(), key=lambda x: x[1], reverse=True)
    palabra_idx = {palabra: idx for idx, (palabra, _) in enumerate(palabras_ordenadas, 2)}
    palabra_idx["<pad>"] = 0
    palabra_idx["<unk>"] = 1
    
    idx_palabra = {idx: palabra for palabra, idx in palabra_idx.items()}
    return palabra_idx, idx_palabra

In [12]:
english_palabra_idx, english_idx_palabra = construir_palabras(english_preprocesado)
spanish_palabra_idx, spanish_idx_palabra = construir_palabras(spanish_preprocesado)

In [13]:
spanish_vocab_size = len(spanish_palabra_idx)
english_vocab_size = len(english_palabra_idx)

spanish_vocab_size, english_vocab_size

### Dataset

In [14]:
from dataset import Traductor
from torch.utils.data import DataLoader

In [15]:
dataset = Traductor(spanish_preprocesado, english_preprocesado, spanish_palabra_idx, english_palabra_idx)
dataset[279325]

In [16]:
english_batch, spanish_batch = dataset[279325]
type(english_batch), type(spanish_batch)


In [17]:
english_batch, spanish_batch

In [18]:
english_batch = english_batch.unsqueeze(0)
spanish_batch = spanish_batch.unsqueeze(0)

In [19]:
english_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in english_batch] # el detach para que no calcule ningun gradiente
spanish_batch = [seq[:MAX_SEQ_LEN].clone().detach() for seq in spanish_batch]

In [20]:
english_batch, spanish_batch

In [21]:
# Agrego pading
import torch
padded_english_tokens = torch.nn.utils.rnn.pad_sequence(english_batch, batch_first=True, padding_value=0)
padded_spanish_tokens = torch.nn.utils.rnn.pad_sequence(spanish_batch, batch_first=True, padding_value=0)

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [22]:
padded_english_tokens, padded_spanish_tokens

### Entrenamiento

In [23]:
dataset

In [24]:
BATCH_SIZE = 64
from dataset import collate_fn
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [25]:
from classes import Transformer
model = Transformer(
    d_model=512, 
    num_heads=8, 
    d_ff=2048,
    num_layers=6,
    input_vocab_size=spanish_vocab_size, 
    target_vocab_size=english_vocab_size, 
    max_len=MAX_SEQ_LEN, 
    dropout=0.1
)

In [26]:
from constants import DEVICE

model.to(DEVICE)

In [27]:
import torch.nn as nn
import torch.optim as optim

criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [28]:
def sentence_to_indices(sentence, word2idx):
    return [word2idx.get(word, word2idx['<unk>']) for word in sentence.split()]

def indices_to_sentence(indices, idx2word):
    return ' '.join([idx2word[idx] for idx in indices if idx in idx2word and idx2word[idx] != '<pad>'])

def translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    model.eval()
    sentence = preprocess_sentence(sentence)
    input_indices = sentence_to_indices(sentence, eng_word2idx)
    input_tensor = torch.tensor(input_indices).unsqueeze(0).to(device)

    # Initialize the target tensor with <sos> token
    tgt_indices = [spa_word2idx['<sos>']]
    tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)

    with torch.no_grad():
        for _ in range(max_len):
            output = model(input_tensor, tgt_tensor)
            output = output.squeeze(0)
            next_token = output.argmax(dim=-1)[-1].item()
            tgt_indices.append(next_token)
            tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0).to(device)
            if next_token == spa_word2idx['<eos>']:
                break

    return indices_to_sentence(tgt_indices, spa_idx2word)


In [30]:
def evaluate_translations(model, sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device='cpu'):
    for sentence in sentences:
        translation = translate_sentence(model, sentence, eng_word2idx, spa_idx2word, max_len, device)
        print(f'Input sentence: {sentence}')
        print(f'Traducción: {translation}')
        print()

# Example sentences to test the translator
test_sentences = [
    "Hola, cómo estás?",
    "Yo soy una IA",
    "La inteligencia artificial es genial",
    "Viva Argentina!"
]

# Assuming the model is trained and loaded
# Set the device to 'cpu' or 'cuda' as needed
#model = model.to(DEVICE)

# Evaluate translations
#evaluate_translations(model, test_sentences, eng_word2idx, spa_idx2word, max_len=MAX_SEQ_LEN, device=device)

In [31]:
from dataset import train
train(model, dataloader, criterion, optimizer, epochs=5)